# 05 — Hypergraph-Native Objective

## Clone the repository

In [40]:
from pathlib import Path

REPO_DIR = Path("/content/tkh-hierarchy-project")

if not REPO_DIR.exists():
    !git clone https://github.com/mohamadghoroobi/tkh-hierarchy-project.git
else:
    print("Repository already cloned.")

Repository already cloned.


In [41]:
%cd /content/tkh-hierarchy-project

/content/tkh-hierarchy-project


## Imports

In [42]:
import json
import math
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

## Paths

In [43]:
PROJECT_DIR = Path("/content/tkh-hierarchy-project")

DATA_DIR = PROJECT_DIR / "data"

SEMANTIC_DIR = (
    PROJECT_DIR
    / "artifacts"
    / "semantic"
)

TKH_PATH = (
    DATA_DIR
    / "tkh_collection10.json"
)

EMBEDDING_PATH = (
    SEMANTIC_DIR
    / "semantic_embeddings.npz"
)

SEMANTIC_METADATA_PATH = (
    SEMANTIC_DIR
    / "semantic_metadata.json"
)

print("TKH:", TKH_PATH)
print("Embeddings:", EMBEDDING_PATH)

TKH: /content/tkh-hierarchy-project/data/tkh_collection10.json
Embeddings: /content/tkh-hierarchy-project/artifacts/semantic/semantic_embeddings.npz


## Load TKH

In [44]:
with open(
    TKH_PATH,
    "r",
    encoding="utf-8"
) as f:
    tkh = json.load(f)


nodes = tkh["nodes"]
hyperedges = tkh["hyperedges"]


print(f"Nodes      : {len(nodes):,}")
print(f"Hyperedges : {len(hyperedges):,}")

Nodes      : 5,798
Hyperedges : 1,429


## Load semantic representation

In [45]:
semantic_artifact = np.load(
    EMBEDDING_PATH
)

semantic_embeddings = (
    semantic_artifact["embeddings"]
    .astype(np.float32)
)

semantic_node_ids = (
    semantic_artifact["node_ids"]
    .astype(str)
    .tolist()
)


print(
    "Embedding matrix:",
    semantic_embeddings.shape
)

print(
    "Node IDs:",
    len(semantic_node_ids)
)

Embedding matrix: (5798, 768)
Node IDs: 5798


In [46]:
node_to_row = {
    node_id: row
    for row, node_id
    in enumerate(semantic_node_ids)
}

assert len(node_to_row) == len(nodes)

print("Semantic node mapping validated.")

Semantic node mapping validated.


## Validate TKH ↔ embedding alignment

In [47]:
tkh_node_ids = {
    node["id"]
    for node in nodes
}

embedding_node_ids = set(
    semantic_node_ids
)

assert (
    tkh_node_ids
    ==
    embedding_node_ids
)

print(
    "TKH and semantic artifact contain identical node IDs."
)

TKH and semantic artifact contain identical node IDs.


This validation is important because otherwise structural and semantic signals could silently refer to different nodes.

## Correct temporal snapshot rule

Use:

$$ v\in V_t \iff first\_seen(v)\le t $$

and:

$$ e\in E_t \iff year(e)\le t $$
	​


In [48]:
SNAPSHOT_YEARS = [
    2020,
    2022,
    2024,
    2026
]

In [49]:
def node_available_at_time(
    node,
    year
):
    first_seen = node.get(
        "first_seen_year"
    )

    return (
        first_seen is not None
        and first_seen <= year
    )

In [50]:
def edge_available_at_time(
    edge,
    year,
    available_node_ids
):
    edge_year = edge.get("year")

    if edge_year is None:
        return False

    if edge_year > year:
        return False

    return all(
        member in available_node_ids
        for member in edge["members"]
    )

## Build snapshots

In [51]:
def build_snapshot(
    nodes,
    hyperedges,
    year
):
    snapshot_nodes = [
        node
        for node in nodes
        if node_available_at_time(
            node,
            year
        )
    ]

    snapshot_node_ids = {
        node["id"]
        for node in snapshot_nodes
    }

    snapshot_edges = [
        edge
        for edge in hyperedges
        if edge_available_at_time(
            edge,
            year,
            snapshot_node_ids
        )
    ]

    return {
        "year": year,
        "nodes": snapshot_nodes,
        "node_ids": snapshot_node_ids,
        "hyperedges": snapshot_edges,
    }

In [52]:
snapshots = {
    year: build_snapshot(
        nodes,
        hyperedges,
        year
    )
    for year in SNAPSHOT_YEARS
}


for year, snapshot in snapshots.items():

    print(
        year,
        f"nodes={len(snapshot['nodes']):,}",
        f"edges={len(snapshot['hyperedges']):,}"
    )

2020 nodes=1,505 edges=374
2022 nodes=2,164 edges=526
2024 nodes=4,164 edges=983
2026 nodes=5,798 edges=1,429


## Our objective

### Hypergraph-native partition objective

For snapshot $H_t=(V_t,E_t)$, let $P$ be a partition of the nodes.

The proposed static objective combines:

1. semantic dispersion inside clusters;
2. native hyperedge fragmentation.

$$
J_\alpha(P)
=
\alpha L_{\mathrm{sem}}(P)
+
(1-\alpha)L_{\mathrm{hyp}}(P)
$$

where $0 \leq \alpha \leq 1$.

Lower values are better.

The semantic component $L_{\mathrm{sem}}(P)$ measures how semantically dispersed the members of each cluster are, using the MPNet embeddings generated in Section 04.

The structural component $L_{\mathrm{hyp}}(P)$ measures how strongly the original hyperedges are fragmented across clusters.

The complete static objective is therefore:

$$
\boxed{
J_\alpha(P)
=
\alpha L_{\mathrm{sem}}(P)
+
(1-\alpha)L_{\mathrm{hyp}}(P)
}
$$

The structural component operates directly on complete hyperedges and never constructs a pairwise graph projection.

## Why use hyperedge entropy?

A structural loss should not only count how many clusters a hyperedge touches; it should also consider how the hyperedge's endpoints are distributed across those clusters.

For example, consider a hyperedge with 65 endpoints.

Case 1:

```text
64 endpoints → cluster A
 1 endpoint  → cluster B


 Case 2:

```text
32 endpoints → cluster A
33 endpoint  → cluster B


And I would also add the semantic loss formula explicitly, because your notebook should not define only the structural side.



## Semantic dispersion term

Each node $v$ has a unit-normalized semantic embedding $x_v$ from Section 04.

For every cluster $C$, define its normalized semantic centroid as

$$
\mu_C
=
\frac{
\sum_{v \in C} x_v
}{
\left\|
\sum_{v \in C} x_v
\right\|_2
}
$$

The semantic loss is the average cosine distance between every node and the centroid of its assigned cluster:

$$
L_{\mathrm{sem}}(P)
=
\frac{1}{|V|}
\sum_{C \in P}
\sum_{v \in C}
\frac{
1 - x_v^\top \mu_C
}{2}
$$

Because both $x_v$ and $\mu_C$ are unit normalized,

$$
x_v^\top \mu_C
=
\cos(x_v,\mu_C)
$$

and therefore lower $L_{\mathrm{sem}}(P)$ indicates more semantically coherent clusters.

The factor of $1/2$ places cosine distance on an approximately $[0,1]$ scale, making it more directly comparable with the normalized hypergraph entropy term.

## Partition validation

We'll represent a partition as:

In [53]:
def validate_partition(
    partition,
    node_ids,
    expected_k=None
):
    node_ids = set(node_ids)

    partition_ids = set(
        partition.keys()
    )

    if partition_ids != node_ids:

        missing = (
            node_ids
            -
            partition_ids
        )

        extra = (
            partition_ids
            -
            node_ids
        )

        raise ValueError(
            f"Partition/node mismatch. "
            f"Missing={len(missing)}, "
            f"Extra={len(extra)}"
        )

    cluster_ids = set(
        partition.values()
    )

    if expected_k is not None:

        if len(cluster_ids) != expected_k:

            raise ValueError(
                f"Expected {expected_k} clusters, "
                f"found {len(cluster_ids)}."
            )

    return True

## Semantic loss

In [54]:
def semantic_partition_loss(
    partition,
    embeddings,
    node_to_row
):
    clusters = defaultdict(list)

    for node_id, cluster_id in partition.items():

        clusters[cluster_id].append(
            node_to_row[node_id]
        )


    total_loss = 0.0
    total_nodes = 0


    for rows in clusters.values():

        X = embeddings[rows]

        centroid = X.mean(
            axis=0
        )

        norm = np.linalg.norm(
            centroid
        )

        if norm == 0:
            raise ValueError(
                "Zero-norm semantic centroid."
            )

        centroid = (
            centroid
            /
            norm
        )

        similarities = (
            X
            @ centroid
        )

        losses = (
            1.0
            -
            similarities
        ) / 2.0

        total_loss += float(
            losses.sum()
        )

        total_nodes += len(rows)


    if total_nodes == 0:
        return 0.0


    return (
        total_loss
        /
        total_nodes
    )

## Hyperedge entropy

In [55]:
def normalized_hyperedge_entropy(
    edge,
    partition,
    num_clusters
):
    members = edge["members"]

    cluster_counts = Counter(
        partition[
            member
        ]
        for member in members
    )


    # Completely internal hyperedge
    if len(cluster_counts) == 1:
        return 0.0


    n = len(members)


    probabilities = np.array(
        [
            count / n
            for count
            in cluster_counts.values()
        ],
        dtype=float
    )


    entropy = -np.sum(
        probabilities
        *
        np.log(probabilities)
    )


    max_blocks = min(
        n,
        num_clusters
    )


    if max_blocks <= 1:
        return 0.0


    max_entropy = math.log(
        max_blocks
    )


    return float(
        entropy
        /
        max_entropy
    )

## Hypergraph structural loss

In [56]:
def hypergraph_partition_loss(
    partition,
    hyperedges,
    edge_weights=None
):
    num_clusters = len(
        set(
            partition.values()
        )
    )


    if edge_weights is None:

        edge_weights = {
            edge["id"]: 1.0
            for edge in hyperedges
        }


    weighted_loss = 0.0
    total_weight = 0.0


    for edge in hyperedges:

        weight = float(
            edge_weights[
                edge["id"]
            ]
        )

        loss = (
            normalized_hyperedge_entropy(
                edge,
                partition,
                num_clusters
            )
        )

        weighted_loss += (
            weight
            *
            loss
        )

        total_weight += weight


    if total_weight == 0:
        return 0.0


    return (
        weighted_loss
        /
        total_weight
    )

This is the key hypergraph-native part.

We are not doing:

hyperedge \
   ↓ \
all node pairs \
   ↓ \
ordinary adjacency matrix


The original hyperedge remains one object throughout the calculation, which directly addresses P4.

## Joint objective

In [57]:
def partition_objective(
    partition,
    snapshot,
    embeddings,
    node_to_row,
    alpha=0.5,
    edge_weights=None
):
    if not (
        0.0
        <= alpha
        <= 1.0
    ):
        raise ValueError(
            "alpha must be in [0, 1]"
        )


    validate_partition(
        partition,
        snapshot["node_ids"]
    )


    semantic_loss = (
        semantic_partition_loss(
            partition,
            embeddings,
            node_to_row
        )
    )


    hypergraph_loss = (
        hypergraph_partition_loss(
            partition,
            snapshot["hyperedges"],
            edge_weights=edge_weights
        )
    )


    total = (
        alpha
        *
        semantic_loss

        +

        (1.0 - alpha)
        *
        hypergraph_loss
    )


    return {
        "total": float(total),

        "semantic":
            float(semantic_loss),

        "hypergraph":
            float(hypergraph_loss),

        "alpha":
            float(alpha),

        "num_clusters":
            len(
                set(
                    partition.values()
                )
            )
    }

## Random exact-K partition helper

Random exact-K partition helper.

In [58]:
def random_partition(
    node_ids,
    k,
    seed=42
):
    node_ids = list(
        sorted(node_ids)
    )

    n = len(node_ids)


    if k < 1 or k > n:
        raise ValueError(
            "k must satisfy 1 <= k <= number of nodes"
        )


    rng = np.random.default_rng(
        seed
    )


    labels = np.arange(n) % k

    rng.shuffle(
        labels
    )


    partition = {
        node_id: int(label)

        for node_id, label
        in zip(
            node_ids,
            labels
        )
    }


    validate_partition(
        partition,
        node_ids,
        expected_k=k
    )


    return partition

## Extreme sanity tests

Use the full 2026 snapshot:

In [59]:
snapshot_2026 = (
    snapshots[2026]
)

ids_2026 = sorted(
    snapshot_2026["node_ids"]
)

One cluster

In [60]:
one_cluster = {
    node_id: 0
    for node_id in ids_2026
}


one_cluster_result = (
    partition_objective(
        one_cluster,
        snapshot_2026,
        semantic_embeddings,
        node_to_row,
        alpha=0.5
    )
)


one_cluster_result

{'total': 0.16490929699963888,
 'semantic': 0.32981859399927776,
 'hypergraph': 0.0,
 'alpha': 0.5,
 'num_clusters': 1}

the resuld is because all hyperedges are internal but unrelated concepts are mixed together.

## Singleton partition

In [61]:
singleton_partition = {
    node_id: i
    for i, node_id
    in enumerate(ids_2026)
}


singleton_result = (
    partition_objective(
        singleton_partition,
        snapshot_2026,
        semantic_embeddings,
        node_to_row,
        alpha=0.5
    )
)


singleton_result

{'total': 0.4999999905884698,
 'semantic': -1.882306046632291e-08,
 'hypergraph': 1.0,
 'alpha': 0.5,
 'num_clusters': 5798}

This is exactly what we want.

The two objective components oppose the two trivial extremes.

## Test realistic resolution budgets

In [62]:
TARGET_CLUSTER_COUNTS = {
    "P0": 12,
    "P1": 60,
    "P2": 300
}

These satisfy the requested coarse-scale constraint \(N_0 $approx \ equal\ to $ 10–15).

## Test random partitions:

In [63]:
sanity_results = []


for level, k in (
    TARGET_CLUSTER_COUNTS.items()
):

    partition = random_partition(
        ids_2026,
        k=k,
        seed=42
    )

    result = partition_objective(
        partition,
        snapshot_2026,
        semantic_embeddings,
        node_to_row,
        alpha=0.5
    )

    sanity_results.append({
        "level": level,
        "k": k,
        **result
    })


sanity_df = pd.DataFrame(
    sanity_results
)

sanity_df

,level,k,total,semantic,hypergraph,alpha,num_clusters
0,P0,12,0.606311,0.328574,0.884048,0.5,12
1,P1,60,0.637817,0.323344,0.952289,0.5,60
2,P2,300,0.647009,0.299344,0.994674,0.5,300


This is not an evaluation.

Random partitions only verify that the objective behaves sensibly.


## α configurations

We should not arbitrarily declare 0.6 semantic / 0.4 structural like the old notebook did.

Instead:

In [65]:
ALPHA_GRID = [
    0.00,
    0.25,
    0.50,
    0.75,
    1.00
]

ALPHA_GRID

[0.0, 0.25, 0.5, 0.75, 1.0]

Interpretation:

α = 0.00 → hypergraph-only

α = 0.25 → structure-heavy

α = 0.50 → balanced

α = 0.75 → semantic-heavy

α = 1.00 → semantic-only

The final value will be chosen using benchmark-blind ablation, not ground_truth.json.

This directly answers the task's requirement that the semantic/structural trade-off be explicit rather than arbitrary.

## Optional relation-balanced weights

For now our primary objective uses uniform edge weights.

But we should support a later ablation because claims is much more frequent than some other relation types.

In [66]:
def build_edge_weights(
    hyperedges,
    scheme="uniform"
):
    if scheme == "uniform":

        return {
            edge["id"]: 1.0
            for edge in hyperedges
        }


    if scheme == "relation_balanced":

        relation_counts = Counter(
            edge["relation_type"]
            for edge in hyperedges
        )


        return {
            edge["id"]:
                1.0
                /
                relation_counts[
                    edge[
                        "relation_type"
                    ]
                ]

            for edge in hyperedges
        }


    raise ValueError(
        f"Unknown edge weighting scheme: {scheme}"
    )

We won't decide which wins now.

That's for ablation later.

## Computational complexity

For a fixed partition:

### Semantic term

Computing cluster centroids and node-to-centroid similarities requires

$$[
O(|V|d)]
$$

where $(d$) is the semantic embedding dimension.

### Hypergraph term

The endpoint distribution of every hyperedge is inspected once:

$$[
O\left(\sum_{e\in E}|e|\right)
]$$

### Full objective

Therefore one complete objective evaluation costs:

$$[
O\left(
|V|d
+
\sum_{e\in E}|e|
\right)
]$$

with no $$(O(|V|^2))$$ pairwise similarity matrix.

Memory is dominated by the semantic embedding matrix and hypergraph incidence data.

The optimization algorithm will be introduced separately in Section 06.

## Formal constraints

## Hard hierarchy constraints

The objective alone does not enforce the hierarchy.

The optimizer will additionally satisfy:

$$[
|P_0| = 12,\quad
|P_1| = 60,\quad
|P_2| = 300
]$$

followed by the singleton node level.

These exact targets are stricter than the assignment's upper-bound display
constraint and therefore satisfy P2 by construction.

Laminarity will be enforced in Section 06 by allowing each finer partition to
refine only its parent supernodes.

Temporal stability is intentionally absent from the current static objective.

This distinction is important:

05 → define static objective\
06 → optimize it into laminar hierarchy\
07 → add temporal coupling

## Save objective configuration


In [67]:
OBJECTIVE_DIR = (
    PROJECT_DIR
    / "artifacts"
    / "objective"
)

OBJECTIVE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [68]:
objective_config = {

    "objective":
        (
            "alpha * semantic_loss "
            "+ (1-alpha) * hypergraph_entropy_loss"
        ),

    "semantic_loss":
        "mean normalized cosine distance to cluster centroid",

    "hypergraph_loss":
        "mean normalized endpoint-distribution entropy per hyperedge",

    "hypergraph_native":
        True,

    "pairwise_projection":
        False,

    "edge_weighting_primary":
        "uniform",

    "edge_weighting_ablation":
        "relation_balanced",

    "alpha_grid":
        ALPHA_GRID,

    "target_clusters":
        TARGET_CLUSTER_COUNTS,

    "snapshot_years":
        SNAPSHOT_YEARS,

    "snapshot_rule": {
        "node":
            "first_seen_year <= snapshot_year",

        "edge":
            (
                "edge.year <= snapshot_year "
                "and all endpoints visible"
            )
    },

    "gold_benchmark_used":
        False
}

In [69]:
OBJECTIVE_CONFIG_PATH = (
    OBJECTIVE_DIR
    / "hypergraph_objective_config.json"
)


with open(
    OBJECTIVE_CONFIG_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        objective_config,
        f,
        indent=2
    )


print(
    "Saved:",
    OBJECTIVE_CONFIG_PATH
)

Saved: /content/tkh-hierarchy-project/artifacts/objective/hypergraph_objective_config.json


## Final summary

In [70]:
print(
    "===== HYPERGRAPH OBJECTIVE SUMMARY ====="
)

print(
    "Semantic term    : centroid cosine dispersion"
)

print(
    "Structural term  : normalized hyperedge entropy"
)

print(
    "Pairwise graph   : False"
)

print(
    "Alpha grid       :", ALPHA_GRID
)

print(
    "Hierarchy targets:",
    TARGET_CLUSTER_COUNTS
)

print(
    "\nSection 05 objective definition passed."
)

===== HYPERGRAPH OBJECTIVE SUMMARY =====
Semantic term    : centroid cosine dispersion
Structural term  : normalized hyperedge entropy
Pairwise graph   : False
Alpha grid       : [0.0, 0.25, 0.5, 0.75, 1.0]
Hierarchy targets: {'P0': 12, 'P1': 60, 'P2': 300}

Section 05 objective definition passed.


## Git Push

In [ ]:
from pathlib import Path

CLEAN = Path("/content/drive/MyDrive/Apply/Germany/ConstructorLabs/03_t1_descriptive_statistics.ipynb")
REPO_FILE = Path(
    "/content/tkh-hierarchy-project/"
    "notebooks/03_t1_descriptive_statistics.ipynb"
)

print("Clean file exists:", CLEAN.exists())
print("Repo file exists :", REPO_FILE.exists())